In [ ]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
import numpy as np
from scipy.spatial.distance import mahalanobis
# T1
legitMean = train[train['Class'] == 0]['Amount'].mean()
fraudAboveMean = train[(train['Class'] == 1) & (train['Amount'] > legitMean)].shape[0]
task1 = int(fraudAboveMean)
# T2
fraud = train[train['Class'] == 1]
fraud = fraud.drop(columns=['id', 'Time', 'Class'])
# num_cols = fraud.select_dtypes(include='float').columns, the dataset already contains just numbers
X = fraud.values

meanVec = X.mean(axis=0)
cov = np.cov(X, rowvar=False)
inv_cov = np.linalg.inv(cov)

distances = [
    mahalanobis(x, meanVec, inv_cov) for x in X
]
mean_dist = np.mean(distances)

In [ ]:
# T3
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import uniform, randint

X_train = train.drop(columns=['id', 'Class'])
y_train = train['Class']
X_test = test.drop(columns=['id'])

pipe = Pipeline([
    ('model', XGBClassifier(random_state=42, tree_method='hist', device='cuda', n_jobs=-1))
])

fraud = train[train['Class'] == 1]
notFraud = train[train['Class'] == 0]
scale_pos_weight = notFraud.shape[0] / fraud.shape[0]

param_dist = {
"model__n_estimators": randint(100, 500),
    "model__max_depth": randint(3, 7),
    "model__learning_rate": uniform(0.01, 0.2),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__scale_pos_weight": [scale_pos_weight]
}

rs = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=15,
    cv=4,
    random_state=42,
    n_jobs=1,
    scoring='f1'
)

rs.fit(X_train, y_train)
model = rs.best_estimator_
predictions = model.predict(X_test)

In [27]:
rows = []
rows.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': int(task1)
})
rows.append({
    'subtaskID': 2,
    'datapointID': 1,
    'answer': round(mean_dist, 2)
})
for id, pred in zip(test['id'], predictions):
    rows.append({
        'subtaskID': 3,
        'datapointID': id,
        'answer': pred.astype(int)
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)